# Second Notebook to use

In [162]:
import os
import pandas as pd
from tmdbv3api import TMDb, Movie, Person
import numpy as np
import requests
import pycountry

In [163]:
TMDB_API_KEY = os.getenv('TMDB_API_KEY')

In [164]:
tmdb = TMDb()
tmdb.api_key = TMDB_API_KEY
tmdb.language = "en"

movie = Movie()

In [165]:
import unicodedata
import re
from useful_funcs import directing_jobs

def normalize_title(s: str) -> str:
    """
    Used to normalize a string if you need an exact comparrison between strings, for example between
    one from scraping and one from an API.
    """
    # 1. Normalize Unicode (NFKC handles compatibility chars)
    s = unicodedata.normalize("NFKC", s)

    # 2. Lowercase
    s = s.lower()

    # 3. Replace all Unicode whitespace with a normal space
    s = re.sub(r"\s+", " ", s)

    # 4. Normalize dash variants (after NFKC, many are already unified)
    s = s.replace("–", "-").replace("—", "-")

    # 5. Strip leading/trailing space
    return s.strip()


def is_movie_in_mc(director_name, movie_name):
     
    """
    Check if given director has made given movie. If yes, returns movie_credits object, else returns None
    """

    movie_name = normalize_title(movie_name)
    person = Person()
    results = person.search(director_name) 
    # directing_jobs = ["Director", "Co-Director", "Assistant Director", 
    #               "Second Unit Director", "Additional Director", "Unit Director", "Segment Director"]
     
    is_in_mc = False
    matching_person = None
    for r in results:       # often gives multiple results
        mc = person.movie_credits(r.id)
        # for m in mc.crew:
        #     print(m.job)
    
        directed_movies = [normalize_title(m.title) for m in mc.crew if m.job in directing_jobs]
        # print(directed_movies)
        # print(movie_name)

        if movie_name in directed_movies:   # found match between movie name and director name!
            is_in_mc = True
            matching_person = r
            break
    
    if not is_in_mc:
        print(f'No director found named {director_name} who directed {movie_name}')
        return None
    
    mc = person.movie_credits(matching_person.id)
    return mc


In [166]:
# put the above in a big function

import ast
from typing import List
def cat_to_list(df: pd.DataFrame, columnnames: List[str]):
    """
    Give a dataframe and a list of columns. Get back the dataframe with the given columns changed from
    string to list of strings instead. Typically used on columns: genre, director, country, language
    """
    for column in columnnames:
        if type(df.loc[0,column]) == str:
            df[column] = df[column].apply(lambda x: ast.literal_eval(x))
    return df

def num_movies_avg_rating(director_name, movie_name):

    """
    Takes director and movie name, returns the folowing: Number of movies directed by director, 
    Average of average score of movies, Average of average score of latest 5 movies
    """

    mc = is_movie_in_mc(director_name=director_name, movie_name=movie_name)             # if movie is in movie_credit of person, returns the mc
    movies = {}

    for m in mc.crew:
        if (m.job in directing_jobs         # a bit hacky
            and hasattr(m, "vote_average")
            and m.vote_average not in (None, 0)
            ):
            if getattr(m, "release_date", None):
                movies[m.id] = {
                    "score": m.vote_average,
                    "release_date": pd.to_datetime(m.release_date, errors="coerce")
                }

    scores = [v['score'] for v in movies.values()]

    last_5_movies = [v['score'] for v in sorted(movies.values(), 
                                                key=lambda x: x['release_date'], reverse=True)[:5]]


    return len(scores), np.average(scores), np.average(last_5_movies)

def get_unique_directors(dframe: pd.DataFrame):
    """
    Takes a dataframe of directors and movies, and returns a list of unique directors as well as a movie
    they have made as an affirmation.
    """

    dframe = cat_to_list(dframe, ['director'])

    unique_directors = []                                                    # the list for the unique directors
    confirmation_movies = []                                                 # movies for confirmation later
    for columnvalue in dframe['director']:
        # columnvalue = columnvalue.replace(' et al', '')                      # ignore et al
        # columnvalue = columnvalue.split(',')                                 # split multiple directors into a list of directors
        # columnvalue = [x.removeprefix(' ') for x in columnvalue]             # remove the extra space

        for director in columnvalue:

            if director not in unique_directors:                             # if director not already in list

                df_exploded = dframe.explode('director')
                movies_by_director = df_exploded[df_exploded['director'] == director]['title'].to_list()

                for movie_title in movies_by_director:
                    if is_movie_in_mc(director, movie_title):                # if a match is found, start appending
                        unique_directors.append(director)
                        confirmation_movies.append(movie_title)
                        break

    return unique_directors, confirmation_movies


In [167]:
def director_db(df_data: pd.DataFrame, df_old_director_db: pd.DataFrame | None = None):
    """
    Give intended training data, returns a database of each director.
    Optionally give old director db to combine with new director db and also speed up by skipping
    """
    df_data = cat_to_list(df_data, ['director'])

    directors, conf_movies = get_unique_directors(df_data)

    if df_old_director_db is not None: # only want to look at directors not already in our old database
        new_direcs_movies = [(x, y) for x, y in zip(directors, conf_movies) if x not in df_old_director_db['director']]
    else:
        new_direcs_movies = zip(directors, conf_movies)

    new_directors = []
    num_movies = []
    avg_movie_score = []
    avg_last_5_movies = []

    for director, movie in new_direcs_movies:

        new_directors.append(director)
        num, avg, avg_last_5 = num_movies_avg_rating(director, movie)
        if num:
            num_movies.append(num)
            avg_movie_score.append(avg)
            avg_last_5_movies.append(avg_last_5)
    
    df_director_db = pd.DataFrame({'director': new_directors, 'director_avg_num_movies': num_movies, 
                             'director_avg_movie_score': avg_movie_score, 
                             'director_avg_last_5_movies': avg_last_5_movies})
    
    if df_old_director_db is not None:
        df_director_db = pd.concat([df_director_db, df_old_director_db], axis=0, ignore_index=True)
    
    return df_director_db

In [168]:
df_scraped = pd.read_csv('initial_dframe.csv')

In [170]:
direc_db1 = director_db(df_scraped)

No director found named Yui Umemoto who directed jujutsu kaisen
No director found named Ryohei Takeshita who directed jujutsu kaisen


# Now for getting top x movies through API

In [171]:
top_url = 'https://api.themoviedb.org/3/movie/top_rated'
top_movies = []

for page in range(1, 11):  # 10 pages × 20 movies = 200
    r = requests.get(
        top_url,
        params={
            "api_key": TMDB_API_KEY,
            "language": "en-US",
            "page": page
        }
    )
    r.raise_for_status()    # raise error if status is not 200
    top_movies.extend(r.json()["results"])

len(top_movies)

200

In [172]:
c=pycountry.countries.get(alpha_3='JPN')
def normalize_country(country):
    """
    Accepts:
    - ISO alpha-2 (US)
    - ISO alpha-3 (USA)
    - English names (United States, Japan)
    Returns:
    - ISO alpha-3 (USA, JPN, ...)
    """
    if not country:
        return None

    try:
        # alpha-2
        c = pycountry.countries.get(alpha_2=country)
        if c:
            return c.alpha_3

        # alpha-3
        c = pycountry.countries.get(alpha_3=country)
        if c:
            return c.alpha_3

        # name
        c = pycountry.countries.search_fuzzy(country)[0]
        return c.alpha_3
    
    except Exception:
        return country

In [173]:
from useful_funcs import directing_jobs
def get_data_from_movie(title):
    result = movie.search(title)
    id = None
    vote_count = 0
    for r in result:
        if r.title == title:
            details = movie.details(r.id)
            if details.vote_count > vote_count:
                vote_count = details.vote_count
                id = r.id

    if id:
        details = movie.details(id)
        credit = movie.credits(id)
        directors = []
        for director_type in directing_jobs:
            if len(directors) >= 2:
                break
            direcs = [
                c['name']
                for c in credit['crew']
                if c['job'] == director_type
            ]
            directors = (directors + direcs)[:2]        # we want 2 directors at most for my setup
        genres = []
        for genre in details.genres:
            genres.append(genre.name)
        release_year = details.release_date[:4]
        vote_average = details.vote_average/2
        country = details.origin_country
        langs = []
        for lang in details.spoken_languages:
            langs.append(lang['english_name'])
        return release_year, vote_average, genres, directors, country, langs

In [174]:
rows = []
movie = Movie()
for film in top_movies:
    title = film['title']
    year, avg_rating, genres, directors, country, language = get_data_from_movie(title=title)
    rows.append({
        'title': title,
        'release_year': year,
        'avg_rating': avg_rating,
        'genre': genres,
        'director': directors,
        'country': country,
        'language': language
    })

In [175]:
df_top_movies = pd.DataFrame(rows)
df_top_movies.head()

,title,release_year,avg_rating,genre,director,country,language
0,The Shawshank Redemption,1994,4.3570,"[Drama, Crime]",[Frank Darabont],[US],[English]
1,The Godfather,1972,4.3430,"[Drama, Crime]",[Francis Ford Coppola],[US],"[English, Italian, Latin]"
2,The Godfather Part II,1974,4.2855,"[Drama, Crime]",[Francis Ford Coppola],[US],"[English, Italian, Latin, Spanish]"
3,Schindler's List,1993,4.2825,"[Drama, History, War]",[Steven Spielberg],[US],"[German, Polish, Hebrew, English]"
4,12 Angry Men,1957,4.2755,[Drama],[Sidney Lumet],[US],[English]


In [176]:
from useful_funcs import normalize_title
df_scraped['title'] = df_scraped['title'].apply(normalize_title)
df_top_movies['title'] = df_top_movies['title'].apply(normalize_title)

In [177]:
# selecting titles in df_top_movies not in df_scraped due to the ~
df_unseen = df_top_movies[~df_top_movies['title'].isin(df_scraped['title'])]
df_unseen.to_csv('unseen_movies.csv', index=False)

In [178]:
df_unseen_load = pd.read_csv('unseen_movies.csv')
direc_db2 = director_db(df_unseen_load, direc_db1)

In [179]:
direc_db2.to_csv('directors.csv', index=False)